In [2]:
from sklearn.metrics import pairwise_distances

# Setup

%load_ext autoreload
%autoreload 2

import os

import torch
import numpy as np
from umap import UMAP
from torch.nn.functional import normalize
from dotenv import load_dotenv
from pandas import read_json
from sklearn.cluster import HDBSCAN
from transformers import AutoModel, AutoTokenizer

from pipelines.multi_service.pooling_layer import mean_pooling

load_dotenv()

print(f"HuggingFace cache directory is {os.environ.get('HF_HOME')}")


def should_be_skipped(class_name: str):
    return (
        class_name.endswith("Test")
        or class_name.endswith("Tests")
        or class_name.endswith("IT")
    )

/home/martin/Projects/Master's Thesis/JENA/JENA.V2/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


HuggingFace cache directory is /media/martin/BigData/datasets/cache/huggingface/


In [9]:
import pandas as pd

# MODEL = "google-bert/bert-base-uncased"
# MODEL = "google-bert/bert-base-cased"
GENERIC_WORDS = [
    "get",
    "set",
    "find by",
    "exists by",
    "exists",
    "update",
    "delete",
    "delete by",
    "create",
    "get all",
    "insert",
    "insert into",
    "find all",
    "find all by",
    "remove",
    "add",
    "edit",
    "save",
    "list",
    "view",
]
# Sort by length descending to ensure "find all by" is matched before "find all"
GENERIC_WORDS.sort(key=len, reverse=True)

MODEL = "microsoft/codebert-base"
# MODEL = "microsoft/graphcodebert-base"

codeBertTokenizer = AutoTokenizer.from_pretrained(MODEL)
codeBertModel = AutoModel.from_pretrained(MODEL)

# bertTokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")
# bertModel = AutoModel.from_pretrained("google-bert/bert-base-uncased")

# input_file = "/media/martin/BigData/datasets/pa165-git/01/multi-service.jsonl"
# output_file = "/media/martin/BigData/datasets/pa165-git/01/multi-service-result.jsonl"

input_file = (
    "/media/martin/BigData/datasets/github-projects/klaw/multi-service-labeled.jsonl"
)
output_file = (
    "/media/martin/BigData/datasets/github-projects/klaw/final-with-centroids.jsonl"
)

# input_file = "/media/martin/BigData/datasets/github-projects/cas/multi-service.json"
# output_file = "/media/martin/BigData/datasets/github-projects/cas/multi-service-result-codebert-combined.jsonl"

input_file = "/media/martin/BigData/datasets/pa165-git/multi-service-filtered-labeled-evaluated.jsonl"
output_file = (
    "/media/martin/BigData/datasets/pa165-git/multi-service-with-distances.jsonl"
)

data_frame = read_json(input_file, lines=True)

# origin_property = "methodSignature"
# evaluation_property = "methodSignatures"


def extract_property(methods: list[dict[str, str]], property: str):
    return list(map(lambda m: m[property], methods))


def remove_words(text: str):
    for word in GENERIC_WORDS:
        if text.startswith(word):
            text = text.removeprefix(word).strip()

    return text


def evaluate(inputs: list[dict[str, str]], _fully_qualified_name: str):
    # class_name = fully_qualified_name.split(".")[-1]
    signature = extract_property(inputs, "signature")
    # cls_with_signature = list(
    #     map(
    #         lambda signature: "class " + class_name + " { \n " + signature + " {} \n}",
    #         signature,
    #     )
    # )
    mean_pooled_signature = get_embeddings(signature)
    # method_names = extract_property(inputs, "name")
    # method_name_split = list(
    #     map(lambda name: remove_words(" ".join(split_camel_case(name))), method_names)
    # )
    # mean_names = get_embeddings(method_name_split)

    signature_normalized = normalize(mean_pooled_signature, p=2, dim=1)
    # name_normalized = normalize(mean_names, p=2, dim=1)

    # combined = torch.cat((signature_normalized, name_normalized), dim=-1)
    mean_pooled_numpy = signature_normalized.cpu().numpy()

    # pca = PCA(n_components=0.95, random_state=42)
    # pca_reduced = pca.fit_transform(mean_pooled_numpy)

    umap = UMAP(
        n_neighbors=5,
        min_dist=0.0,
        n_components=15,
        metric="euclidean",
        random_state=42,
        init="random",
    )
    umap_reduced = umap.fit_transform(mean_pooled_numpy)
    # gap_df = calculate_gap_statistic(
    #     pca_reduced, n_refs=5, max_k=math.ceil(math.sqrt(len(inputs)))
    # )

    clustering_alg = HDBSCAN(
        min_samples=1,
        min_cluster_size=3,
        metric="euclidean",
        allow_single_cluster=True,
        store_centers="medoid",
    )

    labels = clustering_alg.fit_predict(umap_reduced)

    unique_labels = set(labels)
    number_of_clusters = len(unique_labels) - (1 if -1 in unique_labels else 0)
    noise_count = list(labels).count(-1)
    noise_ratio = noise_count / len(labels)
    # return (get_optimal_k_programmatically(gap_df)["optimal_k"], pca_reduced)

    print(labels)
    print(clustering_alg.medoids_)
    medoids = clustering_alg.medoids_
    unique_clusters = unique_labels - {-1}
    avg_distance = 0.0
    if len(unique_clusters) > 1:
        # centroids = []
        # for label in unique_clusters:
        #     centroids.append(np.mean(umap_reduced[labels == label], axis=0))

        if len(unique_clusters) > 1:
            distances = pairwise_distances(medoids, metric="cosine")
            print(distances)
            tri_offset = np.triu_indices(distances.shape[0], k=1)
            unique_distances = distances[tri_offset]

            avg_distance = np.mean(unique_distances)

            # distances = pairwise_distances(centroids, metric="euclidean")
            # tri_offset = np.triu_indices(distances.shape[0], k=1)
            # unique_distances = distances[tri_offset]
            #
            # # 5. Calculate Metrics
            # avg_distance = np.mean(unique_distances)
            # max_distance = np.max(unique_distances)
            # normalized_avg_distance = avg_distance / max_distance

    if all(x == -1 for x in labels):
        number_of_clusters = -1

    return (
        number_of_clusters,
        noise_ratio,
        labels,
        avg_distance,
    )


def get_embeddings(inputs, tokenizer=codeBertTokenizer, model=codeBertModel):
    inputs = list(map(lambda input: "<decoder-only> " + input, inputs))

    tokenized_inputs = tokenizer(
        inputs, return_tensors="pt", padding=True, truncation=True, max_length=512
    )
    with torch.no_grad():
        output = model(**tokenized_inputs)
    attention_mask = tokenized_inputs["attention_mask"]
    return mean_pooling(output, attention_mask)

    # return output.last_hidden_state[:, 0, :]


data_frame["optimal_k"] = pd.Series()
data_frame["noise_ratio"] = pd.Series()
data_frame["cluster"] = pd.Series()
data_frame["avg_dist"] = pd.Series(dtype="float64")
results = []
for index, row in data_frame.iterrows():
    class_name = row["fullyQualifiedName"]
    methods = row["methods"]

    if len(methods) < 10:
        print(f"Skipped small class {class_name}, number of methods {len(methods)}")
        continue

    if should_be_skipped(class_name):
        print(f"Skipped Test class {class_name}, number of methods {len(methods)}")
        continue

    optimal_k, noise_ratio, labels, avg_dist = evaluate(methods, class_name)

    # kmeans = KMeans(n_clusters=optimal_k, random_state=42)
    # cluster = kmeans.fit_predict(pca_reduced)
    data_frame.loc[index, "optimal_k"] = optimal_k
    data_frame.loc[index, "avg_dist"] = avg_dist * 100
    data_frame.loc[index, "noise_ratio"] = noise_ratio
    # data_frame.at[index, "cluster"] = cluster.tolist()
    data_frame.at[index, "cluster"] = labels

    results.append(f"{class_name}: {optimal_k}, {labels}, dist: {avg_dist:.4f}")

for result in results:
    print(result)

data_frame.to_json(output_file, lines=True, orient="records")

Skipped small class cz.muni.fi.pa165.services.beanMapping.BeanMappingServiceImpl, number of methods 3
Skipped small class cz.muni.fi.pa165.services.accountCredentials.PasswordHashingServiceImpl, number of methods 3
Skipped small class cz.muni.fi.pa165.services.itemtype.ItemTypeServiceImpl, number of methods 5
Skipped small class cz.muni.fi.pa165.services.accountCredentials.CredentialsFormatValidatorImpl, number of methods 2
Skipped small class cz.muni.fi.pa165.services.account.AccountServiceImpl, number of methods 7
Skipped small class cz.muni.fi.pa165.services.item.ItemServiceImpl, number of methods 6
Skipped small class cz.muni.fi.pa165.services.image.ImageServiceImpl, number of methods 5
Skipped small class cz.muni.fi.pa165.facades.AccountFacadeImpl, number of methods 7
Skipped small class cz.muni.fi.pa165.services.comment.CommentServiceImpl, number of methods 7
Skipped small class cz.muni.fi.pa165.facades.ItemCreationFacade, number of methods 2
Skipped small class cz.fi.muni.pa165.

/home/martin/Projects/Master's Thesis/JENA/JENA.V2/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


[0 0 0 0 0 0 0 0 0 0 0 0 0]
[[6.80760765 4.80724621 7.91290522 4.43225288 8.34120274 2.53096342
  4.17011786 9.27436924 6.21720028 3.44474316 7.4472909  4.42148018
  3.95347166 4.92707348 4.55130291]]
Skipped small class cz.fi.muni.pa165.winery.service.PersonServiceImpl, number of methods 9
Skipped small class cz.fi.muni.pa165.project.InitialData, number of methods 1
Skipped small class cz.muni.fi.pa165.data.DataLoadingFacadeImpl, number of methods 1
Skipped small class muni.pa165.rest.config.JwtTokenUtil, number of methods 5
Skipped small class muni.pa165.rest.interceptors.JWTFilter, number of methods 0
Skipped small class muni.pa165.api.facade.EventFacade, number of methods 7
Skipped small class muni.pa165.api.facade.CourtFacade, number of methods 5
Skipped small class muni.pa165.api.facade.ParticipantFacade, number of methods 6
Skipped small class cz.muni.fi.pa165.airportmanager.facade.FlightTemplateFacade, number of methods 5
Skipped small class cz.muni.fi.pa165.airportmanager.serv

/home/martin/Projects/Master's Thesis/JENA/JENA.V2/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


[ 0  0 -1 -1  0  0  0  0  0  0  0]
[[1.82514942 2.7712009  8.2039032  5.44341564 1.442963   1.40029001
  1.33752465 9.65287209 9.51103687 3.36014795 6.24074268 3.69124866
  5.66083002 2.95933104 0.92548639]]
Skipped small class cz.idomatojde.services.TimetableChatMessageServiceImpl, number of methods 4
Skipped small class cz.idomatojde.services.OfferPopularityServiceImpl, number of methods 2
Skipped small class cz.idomatojde.services.UserServiceImpl, number of methods 6
Skipped small class cz.idomatojde.services.OfferServiceImpl, number of methods 7
Skipped small class cz.idomatojde.services.facades.TimetableFacadeImpl, number of methods 7
Skipped small class cz.idomatojde.services.facades.CategoryFacadeImpl, number of methods 1


...

/home/martin/Projects/Master's Thesis/JENA/JENA.V2/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


[0 0 0 0 0 0 0 0 0 0 0 0 0 0]
[[6.67411947 3.99454761 6.24233389 4.40876913 4.25476503 4.04291582
  2.34772086 8.97200871 8.23181629 3.16628265 6.78442621 2.85012245
  3.2918005  4.604743   5.99258518]]
Skipped small class cz.muni.fi.pa165.service.impl.RecommendationServiceImpl, number of methods 2
Skipped small class cz.muni.fi.pa165.facade.RatingFacadeImpl, number of methods 7
Skipped small class cz.muni.fi.pa165.service.impl.MovieServiceImpl, number of methods 6
Skipped small class cz.muni.fi.pa165.facade.PersonFacadeImpl, number of methods 7


/home/martin/Projects/Master's Thesis/JENA/JENA.V2/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


[0 0 0 0 0 0 0 0 0 0]
[[6.5861721  1.59521413 4.08331299 7.82051849 2.52133083 0.39316919
  2.70825601 2.46203732 2.84131718 5.47133541 4.12292147 6.94281769
  8.21048641 3.77134347 5.39364529]]
[0 0 0 0 0 0 0 0 0 0]
[[5.63911533 4.71549416 7.33230734 4.0613308  3.47320795 9.74723148
  2.98331451 9.99026299 2.2261436  6.82679367 5.85343361 2.26238751
  2.79037642 5.87942648 3.38563418]]
Skipped small class cz.muni.fi.pa165.service.impl.PersonServiceImpl, number of methods 6
Skipped small class cz.muni.fi.pa165.service.impl.RatingServiceImpl, number of methods 7
Skipped small class cz.muni.fi.pa165.sampledata.SampleDataLoadingFacadeImpl, number of methods 1
Skipped small class cz.muni.pa165.teamwhite.formula1.service.CarServiceImpl, number of methods 5
Skipped small class cz.muni.pa165.teamwhite.formula1.service.DriverServiceImpl, number of methods 5
Skipped small class cz.muni.pa165.teamwhite.formula1.service.facade.DriverFacadeImpl, number of methods 6
Skipped small class cz.muni.pa16

/home/martin/Projects/Master's Thesis/JENA/JENA.V2/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [15]:
import pandas as pd

# input_file = "/media/martin/BigData/datasets/github-projects/klaw/multi-service-final-multiply_signature-remove-words_yes.jsonl"
input_file = (
    "/media/martin/BigData/datasets/github-projects/klaw/final-with-centroids.jsonl"
)
input_file = (
    "/media/martin/BigData/datasets/pa165-git/multi-service-with-distances.jsonl"
)
# input_file = "/media/martin/BigData/datasets/github-projects/cas/multi-service-result-codebert-combined.jsonl"

df = pd.read_json(input_file, lines=True)
df = df.query("optimal_k.notnull()")
more_cluster_lines = df.query("optimal_k > 2")
one_cluster_lines = df.drop(more_cluster_lines.index)


def count_rows_and_print(df: pd.DataFrame, query: str, stat_type: str):
    df_part = df.query(query)
    count = len(df_part)
    print(f"{stat_type}: {count}")
    for _, record in df_part.iterrows():
        print(record["fullyQualifiedName"])
        # print(json.dumps(record["methods"], indent=2))
    print("---")
    return count


false_negatives = count_rows_and_print(
    one_cluster_lines, "label == 1", "False negatives"
)
true_negatives = count_rows_and_print(one_cluster_lines, "label == 0", "True negatives")
false_positives = count_rows_and_print(
    more_cluster_lines, "label == 0", "False positives"
)
true_positives = count_rows_and_print(
    more_cluster_lines, "label == 1", "True positives"
)

precision = true_positives / (true_positives + false_positives)
recall = true_positives / (true_positives + false_negatives)
accuracy = (true_positives + true_negatives) / (
    true_positives + false_positives + false_negatives + true_negatives
)
f1_score = 2 * ((precision * recall) / (precision + recall))

print(f"""
Stats
Precision: {precision}
Recall: {recall}
Accuracy: {accuracy}
F1 score: {f1_score}
-------------
""")

# eval_df = df[df["label"].notna()]
# y_true = eval_df["label"].astype(int)
# y_pred = pd.Series(0, index=eval_df.index)
# y_pred[eval_df.index.isin(more_cluster_lines.index)] = 1

# print("Stats")
# print(classification_report(y_true, y_pred, target_names=["Cohesive", "Multiservice"]))

# class_clusters = defaultdict(dict)
#
# for _, record in more_cluster_lines.iterrows():
#     k = record["optimal_k"]
#     clusters = record["cluster"]
#     methods = record["methods"]
#     class_name = record["fullyQualifiedName"]
#     cohesion = record["lcom4"]
#     noise_ratio = record["noise_ratio"]
#
#     method_clusters = {}
#     for index, cluster in enumerate(clusters):
#         cluster_number_str = str(cluster)
#         method = methods[index]
#         if cluster_number_str in method_clusters:
#             method_clusters[cluster_number_str].append(method)
#         else:
#             method_clusters[cluster_number_str] = [method]
#
#     class_clusters[class_name]["methods"] = method_clusters
#     class_clusters[class_name]["lcom4"] = cohesion
#     class_clusters[class_name]["noise_ratio"] = noise_ratio
#
# for className, metadata in class_clusters.items():
#     print(f"--- Class {className}, LCOM4: {metadata['lcom4']} ---")
#     for cluster, methods in metadata["methods"].items():
#         print(f"Cluster {cluster}: {json.dumps(methods, indent=2)}\n")
#     print()

df

False negatives: 5
cz.fi.muni.pa165.winery.service.BeanMappingServiceImpl
cz.muni.fi.pa165.airportmanager.service.AirplaneServiceImpl
cz.muni.fi.pa165.mvc.config.DataSeeder
cz.muni.fi.pa165.facade.EventFacadeImpl
cz.fi.muni.pa165.hockeymanager.mvc.controllers.TeamController
---
True negatives: 60
cz.idomatojde.services.TimetableServiceImpl
cz.muni.fi.pa165.service.BeerService
cz.muni.fi.pa165.service.facade.BeerFacadeImpl
cz.muni.fi.pa165.service.BeerServiceImpl
muni.pa165.services.UserServiceImpl
cz.fi.muni.pa165.service.OrderServiceImpl
cz.fi.muni.pa165.service.facade.WineFacadeImpl
cz.fi.muni.pa165.service.WineServiceImpl
cz.fi.muni.pa165.service.WineService
cz.fi.muni.pa165.service.UserServiceImpl
cz.fi.muni.pa165.service.UserService
cz.fi.muni.pa165.service.OrderService
cz.fi.muni.pa165.service.facade.OrderFacadeImpl
cz.fi.muni.pa165.service.facade.UserFacadeImpl
cz.fi.muni.pa165.project.service.AirportServiceImpl
cz.fi.muni.pa165.project.facade.AirportFacadeImpl
cz.fi.muni.pa165.

,methods,lcom4,fullyQualifiedName,startLine,label,optimal_k,noise_ratio,cluster,avg_dist
19,"[{'name': 'mapTo', 'signature': 'List<T> mapTo...",4,cz.fi.muni.pa165.winery.service.BeanMappingSer...,22,1,1.0,0.000000,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",0.00000
35,"[{'name': 'find', 'signature': 'Airplane find(...",1,cz.muni.fi.pa165.airportmanager.service.Airpla...,18,1,1.0,0.181818,"[0, 0, -1, -1, 0, 0, 0, 0, 0, 0, 0]",0.00000
42,"[{'name': 'createTimetable', 'signature': 'Tim...",1,cz.idomatojde.services.TimetableServiceImpl,19,0,1.0,0.000000,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",0.00000
45,"[{'name': 'toOfferDTO', 'signature': 'OfferDTO...",2,cz.idomatojde.services.base.MappingServiceImpl,43,1,5.0,0.000000,"[0, 0, 0, 0, 4, 4, 4, 0, 0, 0, 0, 0, 1, 1, 1, ...",8.96476
62,"[{'name': 'create', 'signature': 'Beer create(...",-1,cz.muni.fi.pa165.service.BeerService,16,0,1.0,0.000000,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",0.00000
...,...,...,...,...,...,...,...,...,...
430,"[{'name': 'create', 'signature': 'void create(...",1,cz.muni.fi.pa165.service.MovieServiceImpl,25,0,1.0,0.000000,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",0.00000
451,"[{'name': 'findAll', 'signature': 'List<System...",2,sportsclub.facade.impl.AccountFacadeImpl,33,0,1.0,0.700000,"[0, 0, -1, 0, -1, -1, -1, -1, -1, -1]",0.00000
455,"[{'name': 'create', 'signature': 'MovieDetaile...",1,cz.muni.fi.pa165.facade.MovieFacadeImpl,28,0,1.0,0.000000,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",0.00000
460,"[{'name': 'register', 'signature': 'User regis...",3,cz.muni.fi.pa165.service.impl.UserServiceImpl,24,0,1.0,0.000000,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",0.00000


In [ ]:
from pipelines.shared import input_until_integer

# Labeling script

input_file = "/media/martin/BigData/datasets/github-projects/klaw/multi-service.jsonl"
output_file = (
    "/media/martin/BigData/datasets/github-projects/klaw/multi-service-labeled.jsonl"
)

df = read_json(input_file, lines=True)

df["label"] = pd.Series()

for index, row in df.iterrows():
    name = row["fullyQualifiedName"]
    methods = row["methods"]
    assigned_label = row["label"]
    lcom4 = row["lcom4"]

    label = 0
    if not should_be_skipped(name) and len(methods) > 10:
        print("-" * 10)
        print(f"Class {name} (LCOM4 {lcom4})")
        print("Methods: ")
        for method in methods:
            print(method)

        label = input_until_integer("Enter label: ")

    df.loc[index, "label"] = label
    df.to_json(output_file, lines=True, orient="records")

# Results

## Remove words: yes, names multiply: 1.3

```
False negatives: 1
io.aiven.klaw.service.UtilControllerService
---
True negatives: 10
io.aiven.klaw.MockMethods
io.aiven.klaw.repository.MessageSchemaRepo
io.aiven.klaw.repository.ActivityLogRepo
io.aiven.klaw.controller.AclController
io.aiven.klaw.repository.KwKafkaConnectorRepo
io.aiven.klaw.controller.TopicController
io.aiven.klaw.repository.SchemaRequestRepo
io.aiven.klaw.controller.KafkaConnectController
io.aiven.klaw.service.AnalyticsControllerService
io.aiven.klaw.service.MailUtils
---
False positives: 9
io.aiven.klaw.clusterapi.services.ConfluentCloudApiService
io.aiven.klaw.controller.SchemaRegistryController
io.aiven.klaw.repository.AclRequestsRepo
io.aiven.klaw.repository.AclRepo
io.aiven.klaw.repository.KwKafkaConnectorRequestsRepo
io.aiven.klaw.repository.TopicRequestsRepo
io.aiven.klaw.repository.RegisterInfoRepo
io.aiven.klaw.repository.TopicRepo
io.aiven.klaw.service.ClusterApiService
---
True positives: 15
io.aiven.klaw.clusterapi.utils.ClusterApiUtils
io.aiven.klaw.clusterapi.controller.ClusterApiController
io.aiven.klaw.UtilMethods
io.aiven.klaw.helpers.HandleDbRequests
io.aiven.klaw.controller.TemplateMapController
io.aiven.klaw.controller.UsersTeamsController
io.aiven.klaw.controller.EnvsClustersTenantsController
io.aiven.klaw.helpers.db.rdbms.SelectDataJdbc
io.aiven.klaw.helpers.db.rdbms.HandleDbRequestsJdbc
io.aiven.klaw.helpers.db.rdbms.InsertDataJdbc
io.aiven.klaw.helpers.db.rdbms.DeleteDataJdbc
io.aiven.klaw.service.ServerConfigService
io.aiven.klaw.service.UsersTeamsControllerService
io.aiven.klaw.service.CommonUtilsService
io.aiven.klaw.clusterapi.UtilMethods
---

Stats
Precision: 0.625
Recall: 0.9375
Accuracy: 0.7142857142857143
F1 score: 0.75
```

## Remove words: yes, names multiply: no

```
False negatives: 2
io.aiven.klaw.clusterapi.controller.ClusterApiController
io.aiven.klaw.service.UtilControllerService
---
True negatives: 13
io.aiven.klaw.clusterapi.services.ConfluentCloudApiService
io.aiven.klaw.MockMethods
io.aiven.klaw.controller.SchemaRegistryController
io.aiven.klaw.repository.MessageSchemaRepo
io.aiven.klaw.repository.ActivityLogRepo
io.aiven.klaw.controller.AclController
io.aiven.klaw.repository.KwKafkaConnectorRepo
io.aiven.klaw.controller.TopicController
io.aiven.klaw.repository.SchemaRequestRepo
io.aiven.klaw.controller.KafkaConnectController
io.aiven.klaw.service.AnalyticsControllerService
io.aiven.klaw.service.MailUtils
io.aiven.klaw.service.ClusterApiService
---
False positives: 6
io.aiven.klaw.repository.AclRequestsRepo
io.aiven.klaw.repository.AclRepo
io.aiven.klaw.repository.KwKafkaConnectorRequestsRepo
io.aiven.klaw.repository.TopicRequestsRepo
io.aiven.klaw.repository.RegisterInfoRepo
io.aiven.klaw.repository.TopicRepo
---
True positives: 14
io.aiven.klaw.clusterapi.utils.ClusterApiUtils
io.aiven.klaw.UtilMethods
io.aiven.klaw.helpers.HandleDbRequests
io.aiven.klaw.controller.TemplateMapController
io.aiven.klaw.controller.UsersTeamsController
io.aiven.klaw.controller.EnvsClustersTenantsController
io.aiven.klaw.helpers.db.rdbms.SelectDataJdbc
io.aiven.klaw.helpers.db.rdbms.HandleDbRequestsJdbc
io.aiven.klaw.helpers.db.rdbms.InsertDataJdbc
io.aiven.klaw.helpers.db.rdbms.DeleteDataJdbc
io.aiven.klaw.service.ServerConfigService
io.aiven.klaw.service.UsersTeamsControllerService
io.aiven.klaw.service.CommonUtilsService
io.aiven.klaw.clusterapi.UtilMethods
---

Stats
Precision: 0.7
Recall: 0.875
Accuracy: 0.7714285714285715
F1 score: 0.7777777777777777
-------------
```

## Remove words: no, names multiply: no

```
False negatives: 2
io.aiven.klaw.clusterapi.controller.ClusterApiController
io.aiven.klaw.service.UtilControllerService
---
True negatives: 10
io.aiven.klaw.clusterapi.services.ConfluentCloudApiService
io.aiven.klaw.MockMethods
io.aiven.klaw.repository.MessageSchemaRepo
io.aiven.klaw.controller.AclController
io.aiven.klaw.repository.KwKafkaConnectorRepo
io.aiven.klaw.controller.TopicController
io.aiven.klaw.controller.KafkaConnectController
io.aiven.klaw.service.AnalyticsControllerService
io.aiven.klaw.service.MailUtils
io.aiven.klaw.service.ClusterApiService
---
False positives: 9
io.aiven.klaw.controller.SchemaRegistryController
io.aiven.klaw.repository.AclRequestsRepo
io.aiven.klaw.repository.AclRepo
io.aiven.klaw.repository.ActivityLogRepo
io.aiven.klaw.repository.KwKafkaConnectorRequestsRepo
io.aiven.klaw.repository.TopicRequestsRepo
io.aiven.klaw.repository.SchemaRequestRepo
io.aiven.klaw.repository.RegisterInfoRepo
io.aiven.klaw.repository.TopicRepo
---
True positives: 14
io.aiven.klaw.clusterapi.utils.ClusterApiUtils
io.aiven.klaw.UtilMethods
io.aiven.klaw.helpers.HandleDbRequests
io.aiven.klaw.controller.TemplateMapController
io.aiven.klaw.controller.UsersTeamsController
io.aiven.klaw.controller.EnvsClustersTenantsController
io.aiven.klaw.helpers.db.rdbms.SelectDataJdbc
io.aiven.klaw.helpers.db.rdbms.HandleDbRequestsJdbc
io.aiven.klaw.helpers.db.rdbms.InsertDataJdbc
io.aiven.klaw.helpers.db.rdbms.DeleteDataJdbc
io.aiven.klaw.service.ServerConfigService
io.aiven.klaw.service.UsersTeamsControllerService
io.aiven.klaw.service.CommonUtilsService
io.aiven.klaw.clusterapi.UtilMethods
---

Stats
Precision: 0.6086956521739131
Recall: 0.875
Accuracy: 0.6857142857142857
F1 score: 0.717948717948718
-------------
```

## Remove words: no, names multiply: 1.3
```
False negatives: 2
io.aiven.klaw.clusterapi.controller.ClusterApiController
io.aiven.klaw.service.UtilControllerService
---
True negatives: 9
io.aiven.klaw.clusterapi.services.ConfluentCloudApiService
io.aiven.klaw.MockMethods
io.aiven.klaw.repository.ActivityLogRepo
io.aiven.klaw.controller.AclController
io.aiven.klaw.repository.KwKafkaConnectorRepo
io.aiven.klaw.controller.TopicController
io.aiven.klaw.controller.KafkaConnectController
io.aiven.klaw.service.AnalyticsControllerService
io.aiven.klaw.service.MailUtils
---
False positives: 10
io.aiven.klaw.controller.SchemaRegistryController
io.aiven.klaw.repository.AclRequestsRepo
io.aiven.klaw.repository.MessageSchemaRepo
io.aiven.klaw.repository.AclRepo
io.aiven.klaw.repository.KwKafkaConnectorRequestsRepo
io.aiven.klaw.repository.TopicRequestsRepo
io.aiven.klaw.repository.SchemaRequestRepo
io.aiven.klaw.repository.RegisterInfoRepo
io.aiven.klaw.repository.TopicRepo
io.aiven.klaw.service.ClusterApiService
---
True positives: 14
io.aiven.klaw.clusterapi.utils.ClusterApiUtils
io.aiven.klaw.UtilMethods
io.aiven.klaw.helpers.HandleDbRequests
io.aiven.klaw.controller.TemplateMapController
io.aiven.klaw.controller.UsersTeamsController
io.aiven.klaw.controller.EnvsClustersTenantsController
io.aiven.klaw.helpers.db.rdbms.SelectDataJdbc
io.aiven.klaw.helpers.db.rdbms.HandleDbRequestsJdbc
io.aiven.klaw.helpers.db.rdbms.InsertDataJdbc
io.aiven.klaw.helpers.db.rdbms.DeleteDataJdbc
io.aiven.klaw.service.ServerConfigService
io.aiven.klaw.service.UsersTeamsControllerService
io.aiven.klaw.service.CommonUtilsService
io.aiven.klaw.clusterapi.UtilMethods
---

Stats
Precision: 0.5833333333333334
Recall: 0.875
Accuracy: 0.6571428571428571
F1 score: 0.7000000000000001
```

## Remove words: yes, signature multiply: 1.3

```
False negatives: 1
io.aiven.klaw.clusterapi.controller.ClusterApiController
---
True negatives: 10
io.aiven.klaw.clusterapi.services.ConfluentCloudApiService
io.aiven.klaw.MockMethods
io.aiven.klaw.controller.SchemaRegistryController
io.aiven.klaw.repository.MessageSchemaRepo
io.aiven.klaw.repository.KwKafkaConnectorRepo
io.aiven.klaw.controller.TopicController
io.aiven.klaw.repository.SchemaRequestRepo
io.aiven.klaw.controller.KafkaConnectController
io.aiven.klaw.service.MailUtils
io.aiven.klaw.service.ClusterApiService
---
False positives: 9
io.aiven.klaw.repository.AclRequestsRepo
io.aiven.klaw.repository.AclRepo
io.aiven.klaw.repository.ActivityLogRepo
io.aiven.klaw.controller.AclController
io.aiven.klaw.repository.KwKafkaConnectorRequestsRepo
io.aiven.klaw.repository.TopicRequestsRepo
io.aiven.klaw.repository.RegisterInfoRepo
io.aiven.klaw.repository.TopicRepo
io.aiven.klaw.service.AnalyticsControllerService
---
True positives: 15
io.aiven.klaw.clusterapi.utils.ClusterApiUtils
io.aiven.klaw.UtilMethods
io.aiven.klaw.helpers.HandleDbRequests
io.aiven.klaw.controller.TemplateMapController
io.aiven.klaw.controller.UsersTeamsController
io.aiven.klaw.controller.EnvsClustersTenantsController
io.aiven.klaw.helpers.db.rdbms.SelectDataJdbc
io.aiven.klaw.helpers.db.rdbms.HandleDbRequestsJdbc
io.aiven.klaw.helpers.db.rdbms.InsertDataJdbc
io.aiven.klaw.helpers.db.rdbms.DeleteDataJdbc
io.aiven.klaw.service.ServerConfigService
io.aiven.klaw.service.UsersTeamsControllerService
io.aiven.klaw.service.UtilControllerService
io.aiven.klaw.service.CommonUtilsService
io.aiven.klaw.clusterapi.UtilMethods
---

Stats
Precision: 0.625
Recall: 0.9375
Accuracy: 0.7142857142857143
F1 score: 0.75
```


# TODO
- Extract LCOM4 for Klaw with delombok and newest jena
- Try combination of optimal_k > 2 and lcom4
- Try graphcodebert for method bodies + signatures

In [12]:
input_file = (
    "/media/martin/BigData/datasets/pa165-git/multi-service-filtered-labeled.jsonl"
)

df = read_json(input_file, lines=True)

df.query("label == 0 and len(methods) > 10")

ValueError: "len" is not a supported function